# 5.10 Regularization

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 5: Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-05-linear-regression/05.10_regularization.ipynb)

Companion code for Section 5.10 of the book. Run the cells in order from top to bottom.

**Example 5.10.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_california_housing

# ── Illustrate regularisation with high-dimensional data ──────────
np.random.seed(42)
n, p = 200, 50    # 200 samples, 50 features (many irrelevant)
n_informative = 10  # Only 10 features actually matter

X = np.random.randn(n, p)
true_coef = np.zeros(p)
true_coef[:n_informative] = np.random.choice([-3,-2,-1,1,2,3], n_informative)
y = X @ true_coef + np.random.normal(0, 1, n)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

# ── Fit all four models ───────────────────────────────────────────
models = {
    'OLS':       LinearRegression(),
    'Ridge':     Ridge(alpha=1.0),
    'Lasso':     Lasso(alpha=0.1, max_iter=5000),
    'ElasticNet':ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
}

results = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Regularisation Comparison: OLS vs Ridge vs Lasso vs ElasticNet',
            fontsize=13, fontweight='bold')

for idx, (name, model) in enumerate(models.items()):
    model.fit(X_tr, y_train)
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)

    coef = model.coef_
    n_zeros    = np.sum(np.abs(coef) < 1e-6)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_tr_pred))
    test_rmse  = np.sqrt(mean_squared_error(y_test, y_te_pred))

    results[name] = {
        'train_rmse': train_rmse, 'test_rmse': test_rmse,
        'n_zeros': n_zeros, 'coef': coef,
        'coef_norm': np.linalg.norm(coef)
    }

    # Plot coefficients
    ax = axes[idx//2, idx%2]
    colors = ['#E53935' if abs(c)>0.1 else '#BDBDBD' for c in coef]
    ax.bar(range(p), coef, color=colors, edgecolor='white', width=0.8)
    ax.scatter(range(n_informative), true_coef[:n_informative],
              color='black', marker='*', s=100, zorder=5, label='True coef')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('Feature Index'); ax.set_ylabel('Coefficient Value')
    ax.set_title(f'{name}\nTest RMSE={test_rmse:.3f}  Zeros={n_zeros}/{p}',
               fontsize=11)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('regularisation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ────────────────────────────────────────────────
print(f'\n=== Regularisation Summary ===')
print(f'True model: {n_informative} non-zero coefficients out of {p}')
print(f'\n{"Model":<12} {"Train RMSE":>11} {"Test RMSE":>10} {"Zeros":>8} {"‖β‖":>8}')
print('-'*55)
for name, r in results.items():
    print(f'{name:<12} {r["train_rmse"]:>11.4f} {r["test_rmse"]:>10.4f} {r["n_zeros"]:>8}/{p}  {r["coef_norm"]:>7.3f}')

# ── Regularisation path: how coefficients change with λ ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Regularisation Paths - Coefficient Shrinkage', fontsize=13, fontweight='bold')

alphas_ridge = np.logspace(-4, 4, 100)
alphas_lasso  = np.logspace(-4, 1, 100)

# Ridge path
ax = axes[0]
ridge_coefs = []
for a in alphas_ridge:
    ridge_coefs.append(Ridge(alpha=a).fit(X_tr, y_train).coef_)
ridge_coefs = np.array(ridge_coefs)
for j in range(p):
    alpha_val = 0.6 if j < n_informative else 0.1
    lw = 2 if j < n_informative else 0.5
    ax.plot(alphas_ridge, ridge_coefs[:,j], alpha=alpha_val, linewidth=lw,
           color='#1976D2' if j < n_informative else '#BDBDBD')
ax.set_xscale('log'); ax.set_xlabel('λ (Regularisation Strength)')
ax.set_ylabel('Coefficient Value')
ax.set_title('Ridge Regression Path\n(Blue = informative features)')
ax.axvline(x=1, color='red', linestyle='--', label='λ=1')
ax.legend(); ax.grid(alpha=0.3)

# Lasso path
ax = axes[1]
lasso_coefs = []
for a in alphas_lasso:
    lasso_coefs.append(Lasso(alpha=a, max_iter=5000).fit(X_tr, y_train).coef_)
lasso_coefs = np.array(lasso_coefs)
for j in range(p):
    alpha_val = 0.8 if j < n_informative else 0.15
    lw = 2 if j < n_informative else 0.5
    ax.plot(alphas_lasso, lasso_coefs[:,j], alpha=alpha_val, linewidth=lw,
           color='#E53935' if j < n_informative else '#BDBDBD')
ax.set_xscale('log'); ax.set_xlabel('λ (Regularisation Strength)')
ax.set_ylabel('Coefficient Value')
ax.set_title('Lasso Regression Path\n(Features zeroed out as λ increases)')
ax.axvline(x=0.1, color='red', linestyle='--', label='λ=0.1')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('regularisation_paths.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Hyperparameter tuning with cross-validation ───────────────────
from sklearn.linear_model import RidgeCV, LassoCV

# RidgeCV: built-in efficient cross-validation for alpha selection
alphas = np.logspace(-4, 4, 100)
ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_tr, y_train)
print(f'\nRidgeCV optimal alpha: {ridge_cv.alpha_:.6f}')
print(f'RidgeCV Test RMSE:    {np.sqrt(mean_squared_error(y_test, ridge_cv.predict(X_te))):.4f}')

# LassoCV: coordinate descent CV
lasso_cv = LassoCV(cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_tr, y_train)
print(f'\nLassoCV optimal alpha: {lasso_cv.alpha_:.6f}')
print(f'LassoCV Test RMSE:    {np.sqrt(mean_squared_error(y_test, lasso_cv.predict(X_te))):.4f}')
print(f'LassoCV Non-zero coef: {(np.abs(lasso_cv.coef_) > 1e-6).sum()}/{p}')